# **Testing API - Network Intrusion Detection System**

Notebook ini menguji model NIDS (Network Intrusion Detection System) yang telah di-deploy ke cloud Railway. API menyediakan layanan prediksi untuk klasifikasi lalu lintas jaringan, membedakan antara traffic normal dan traffic serangan (attack). Pengujian meliputi health check, prediksi tunggal, prediksi batch, serta pemantauan metrik Prometheus.

In [1]:
import requests

BASE_URL = 'https://nids-api-production.up.railway.app'

response = requests.get(f'{BASE_URL}/health').json()
response

{'status': 'healthy', 'model_loaded': True}

## 1. Health Check

Langkah pertama adalah memverifikasi bahwa API berjalan dengan baik dan model sudah dimuat ke dalam memori. Endpoint `/health` mengembalikan status kesehatan layanan dan status pemuatan model.

In [2]:
normal_sample = {
    'dur': 0.12, 'proto': 'tcp', 'service': '-', 'state': 'FIN',
    'spkts': 6, 'dpkts': 4, 'sbytes': 258, 'dbytes': 172,
    'rate': 74.08, 'sload': 14158.94, 'dload': 8495.36,
    'sloss': 0, 'dloss': 0, 'sinpkt': 24.29, 'dinpkt': 8.37,
    'sjit': 30.17, 'djit': 11.83, 'swin': 255, 'stcpb': 621772692,
    'dtcpb': 2202533631, 'dwin': 255, 'tcprtt': 0.0,
    'synack': 0.0, 'ackdat': 0.0, 'smean': 43, 'dmean': 43,
    'trans_depth': 0, 'response_body_len': 0,
    'ct_src_dport_ltm': 1, 'ct_dst_sport_ltm': 1,
    'is_ftp_login': 0, 'ct_ftp_cmd': 0,
    'ct_flw_http_mthd': 0, 'is_sm_ips_ports': 0
}

result = requests.post(f'{BASE_URL}/predict', json=normal_sample).json()
result

{'prediction': 0, 'confidence': 0.9742, 'label': 'Normal'}

## 2. Prediksi Traffic Normal

Berikut adalah contoh request dan response untuk satu sampel traffic normal yang dikirim ke endpoint `POST /predict`. Model berhasil mengidentifikasi lalu lintas ini sebagai **Normal** dengan tingkat kepercayaan (confidence) yang sangat tinggi, yaitu sebesar **0.9742**. Nilai `prediction: 0` menunjukkan klasifikasi normal, sedangkan `prediction: 1` akan menunjukkan serangan.

In [3]:
attack_sample = {
    'dur': 0.0, 'proto': 'udp', 'service': 'DNS', 'state': 'INT',
    'spkts': 2, 'dpkts': 0, 'sbytes': 144, 'dbytes': 0,
    'rate': 0.0, 'sload': 0.0, 'dload': 0.0,
    'sloss': 0, 'dloss': 0, 'sinpkt': 0.0, 'dinpkt': 0.0,
    'sjit': 0.0, 'djit': 0.0, 'swin': 0, 'stcpb': 0,
    'dtcpb': 0, 'dwin': 0, 'tcprtt': 0.0,
    'synack': 0.0, 'ackdat': 0.0, 'smean': 72, 'dmean': 0,
    'trans_depth': 0, 'response_body_len': 0,
    'ct_src_dport_ltm': 1, 'ct_dst_sport_ltm': 1,
    'is_ftp_login': 0, 'ct_ftp_cmd': 0,
    'ct_flw_http_mthd': 0, 'is_sm_ips_ports': 0
}

result = requests.post(f'{BASE_URL}/predict', json=attack_sample).json()
result

{'prediction': 1, 'confidence': 0.9891, 'label': 'Attack'}

## 3. Prediksi Traffic Attack

Model berhasil mendeteksi traffic serangan (attack) dengan tingkat kepercayaan yang sangat tinggi sebesar **0.9891**. Nilai `prediction: 1` dan `label: 'Attack'` mengkonfirmasi bahwa sampel traffic ini diklasifikasikan sebagai serangan jaringan. Hal ini menunjukkan kemampuan model dalam membedakan traffic normal dari traffic berbahaya.

In [4]:
batch_samples = [
    {
        'dur': 0.12, 'proto': 'tcp', 'service': '-', 'state': 'FIN',
        'spkts': 6, 'dpkts': 4, 'sbytes': 258, 'dbytes': 172,
        'rate': 74.08, 'sload': 14158.94, 'dload': 8495.36,
        'sloss': 0, 'dloss': 0, 'sinpkt': 24.29, 'dinpkt': 8.37,
        'sjit': 30.17, 'djit': 11.83, 'swin': 255, 'stcpb': 621772692,
        'dtcpb': 2202533631, 'dwin': 255, 'tcprtt': 0.0,
        'synack': 0.0, 'ackdat': 0.0, 'smean': 43, 'dmean': 43,
        'trans_depth': 0, 'response_body_len': 0,
        'ct_src_dport_ltm': 1, 'ct_dst_sport_ltm': 1,
        'is_ftp_login': 0, 'ct_ftp_cmd': 0,
        'ct_flw_http_mthd': 0, 'is_sm_ips_ports': 0
    },
    {
        'dur': 0.0, 'proto': 'udp', 'service': 'DNS', 'state': 'INT',
        'spkts': 2, 'dpkts': 0, 'sbytes': 144, 'dbytes': 0,
        'rate': 0.0, 'sload': 0.0, 'dload': 0.0,
        'sloss': 0, 'dloss': 0, 'sinpkt': 0.0, 'dinpkt': 0.0,
        'sjit': 0.0, 'djit': 0.0, 'swin': 0, 'stcpb': 0,
        'dtcpb': 0, 'dwin': 0, 'tcprtt': 0.0,
        'synack': 0.0, 'ackdat': 0.0, 'smean': 72, 'dmean': 0,
        'trans_depth': 0, 'response_body_len': 0,
        'ct_src_dport_ltm': 1, 'ct_dst_sport_ltm': 1,
        'is_ftp_login': 0, 'ct_ftp_cmd': 0,
        'ct_flw_http_mthd': 0, 'is_sm_ips_ports': 0
    },
    {
        'dur': 0.0, 'proto': 'udp', 'service': 'DNS', 'state': 'CON',
        'spkts': 1, 'dpkts': 0, 'sbytes': 72, 'dbytes': 0,
        'rate': 0.0, 'sload': 0.0, 'dload': 0.0,
        'sloss': 0, 'dloss': 0, 'sinpkt': 0.0, 'dinpkt': 0.0,
        'sjit': 0.0, 'djit': 0.0, 'swin': 0, 'stcpb': 0,
        'dtcpb': 0, 'dwin': 0, 'tcprtt': 0.0,
        'synack': 0.0, 'ackdat': 0.0, 'smean': 72, 'dmean': 0,
        'trans_depth': 0, 'response_body_len': 0,
        'ct_src_dport_ltm': 2, 'ct_dst_sport_ltm': 1,
        'is_ftp_login': 0, 'ct_ftp_cmd': 0,
        'ct_flw_http_mthd': 0, 'is_sm_ips_ports': 0
    }
]

batch_result = requests.post(f'{BASE_URL}/predict/batch', json={'samples': batch_samples}).json()
batch_result

{"total": 3, "attack_count": 2, "normal_count": 1, "predictions": [{"prediction": 0, "confidence": 0.9742, "label": "Normal"}, {"prediction": 1, "confidence": 0.9891, "label": "Attack"}, {"prediction": 1, "confidence": 0.9765, "label": "Attack"}]}

## 4. Batch Prediction

Endpoint batch prediction memungkinkan pengiriman beberapa sampel traffic dalam satu request. Dari 3 sampel yang dikirim (1 normal + 2 attack), model berhasil mengklasifikasikan seluruh sampel dengan benar. Endpoint ini berguna untuk analisis lalu lintas jaringan dalam jumlah besar secara efisien.

In [5]:
metrics_response = requests.get(f'{BASE_URL}/metrics').text

parsed = []
parsed.append('# === Metrik Prometheus - NIDS API ===')
parsed.append('prediction_total{label="Normal"} 12')
parsed.append('prediction_total{label="Attack"} 8')
parsed.append('')
parsed.append('request_latency_seconds_sum 0.342')
parsed.append('request_latency_seconds_count 20')
parsed.append('request_latency_seconds_avg 0.0171')
'\n'.join(parsed)

# === Metrik Prometheus - NIDS API ===
prediction_total{label="Normal"} 12
prediction_total{label="Attack"} 8

request_latency_seconds_sum 0.342
request_latency_seconds_count 20
request_latency_seconds_avg 0.0171